In [1]:
#CE 311 K Project 
# import libraries
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

Environmental Engineering Calculations & Graphing
PFR + CSTR
Unit Converter
Maybe BOD?

In [13]:
from scipy.optimize import fsolve
import numpy as np

def cstr_calculator(Qins=None, Cins=None, Qouts=None, Cout=None, V=None, HRT=None):
    """
    Solves the steady-state CSTR mass balance for any single unknown.
        Flow : Σ(Qin_i) = Σ(Qout_j)
        Mass : Σ(Qin_i * Cin_i) = Cout * Σ(Qout_j)
        HRT  : HRT = V / Σ(Qout_j)
    Parameters:
        Qins  : list of floats  — inlet flow rates
        Cins  : list of floats  — inlet concentrations (same length as Qins)
        Qouts : list of floats  — outlet flow rates
        Cout  : float           — outlet concentration (shared by all outlets)
        V     : float           — reactor volume
        HRT   : float           — hydraulic retention time (V / ΣQout)
    """

    # --- V / HRT are linked via HRT = V / ΣQout ---
    # Defer computing whichever is missing until after Qouts is known
    v_unknown   = V   is None
    hrt_unknown = HRT is None

    if v_unknown and hrt_unknown:
        raise ValueError(
            "Provide at least one of V or HRT. "
            "V cannot be solved from mass balance alone (no reaction term). "
            "Supply HRT to back-calculate V."
        )

    unknowns = {
        "Qins":  Qins  is None,
        "Cins":  Cins  is None,
        "Qouts": Qouts is None,
        "Cout":  Cout  is None,
    }

    num_unknowns = sum(unknowns.values())
    if num_unknowns != 1:
        raise ValueError(f"Exactly 1 of (Qins, Cins, Qouts, Cout) must be None (got {num_unknowns}).")

    if Qins is not None and Cins is not None and len(Qins) != len(Cins):
        raise ValueError(f"Qins (len={len(Qins)}) and Cins (len={len(Cins)}) must be the same length.")

    # --- Solver ---
    def equation(x):
        _Qins  = x.tolist()  if unknowns["Qins"]  else Qins
        _Cins  = x.tolist()  if unknowns["Cins"]  else Cins
        _Qouts = x.tolist()  if unknowns["Qouts"] else Qouts
        _Cout  = float(x[0]) if unknowns["Cout"]  else Cout

        total_Qin  = sum(_Qins)
        total_Qout = sum(_Qouts)
        inlet_load = sum(q * c for q, c in zip(_Qins, _Cins))

        if unknowns["Qouts"]:
            return total_Qin - total_Qout           # flow balance
        return inlet_load - _Cout * total_Qout      # mass balance

    x0 = [1.0]
    solution, info, ier, msg = fsolve(equation, x0, full_output=True)

    if ier != 1:
        raise RuntimeError(f"Solver failed to converge: {msg}")

    solved = float(solution[0])

    # --- Reconstruct full variable set ---
    _Qins  = Qins  if not unknowns["Qins"]  else [solved]
    _Cins  = Cins  if not unknowns["Cins"]  else [solved]
    _Qouts = Qouts if not unknowns["Qouts"] else [solved]
    _Cout  = Cout  if not unknowns["Cout"]  else solved

    total_Qin  = sum(_Qins)
    total_Qout = sum(_Qouts)

    # --- Flow balance check ---
    if not unknowns["Qins"] and not unknowns["Qouts"]:
        if not np.isclose(total_Qin, total_Qout, rtol=0.01):
            raise ValueError(
                f"Flow imbalance: sum(Qins)={total_Qin} ≠ sum(Qouts)={total_Qout}. "
                "No steady-state solution exists."
            )

    # --- V / HRT resolution (now that Qouts is known) ---
    if not v_unknown and not hrt_unknown:
        pass  # both provided, use as-is
    elif v_unknown:
        V = HRT * total_Qout
    elif hrt_unknown:
        HRT = V / total_Qout

    return {
        "Qins":         _Qins,
        "Cins":         _Cins,
        "Qouts":        _Qouts,
        "Cout":         float(_Cout),
        "V":            float(V),
        "HRT":          float(HRT),
        "total_Qin":    float(total_Qin),
        "total_Qout":   float(total_Qout),
        "solved_for":   [k for k, v in unknowns.items() if v][0],
        "solved_value": float(solved),
    }


def cstr_sweep(sweep_param, sweep_values, **kwargs):
    """
    Runs cstr_calculator across a range of values for one parameter,
    returning results as numpy arrays ready for plotting.

    Parameters:
        sweep_param  : str   — name of the parameter to vary
                               ('Qins', 'Cins', 'Qouts', 'Cout', 'V', 'HRT')
        sweep_values : array — values to sweep over
        **kwargs     : all other cstr_calculator parameters (fixed)

    Returns:
        dict of numpy arrays, one value per sweep point:
    """

    list_params = {"Qins", "Cins", "Qouts"}

    records = {
        "sweep_values": [],
        "Cout":         [],
        "V":            [],
        "HRT":          [],
        "total_Qin":    [],
        "total_Qout":   [],
        "solved_values":[],
    }

    for val in sweep_values:
        params = dict(kwargs)

        # List params wrap the scalar in a list to match expected input type
        if sweep_param in list_params:
            params[sweep_param] = [val]
        else:
            params[sweep_param] = val

        result = cstr_calculator(**params)

        records["sweep_values"].append(val)
        records["Cout"].append(result["Cout"])
        records["V"].append(result["V"])
        records["HRT"].append(result["HRT"])
        records["total_Qin"].append(result["total_Qin"])
        records["total_Qout"].append(result["total_Qout"])
        records["solved_values"].append(result["solved_value"])

    return {k: np.array(v) for k, v in records.items()}

a = cstr_calculator(Qins=[10, 20], Cins=[100, 200], Qouts=[30], Cout=None, V=500)
print(a)

{'Qins': [10, 20], 'Cins': [100, 200], 'Qouts': [30], 'Cout': 166.66666666666666, 'V': 500.0, 'HRT': 16.666666666666668, 'total_Qin': 30.0, 'total_Qout': 30.0, 'solved_for': 'Cout', 'solved_value': 166.66666666666666}


In [ ]:
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp
from scipy.special import factorial
import numpy as np
#updated CSTR in Series 
def cstr_series_calc(Co=None, total_HRT=None, n=None, volumes=None, Q=None, Cn=None, t=None):
    """
    Solves the tanks-in-series RTD model for CSTRs in series.

    Two modes depending on inputs:
        Equal volumes   : analytical formula using total_HRT and n
        Unequal volumes : ODE simulation using volumes list + Q

    Analytical formula (equal volumes, pulse tracer input):
        HRT_i = total_HRT / n
        C(t) = Co * (1/HRT_i) * (t/HRT_i)^(n-1) / (n-1)! * exp(-t/HRT_i)

    ODE mass balance (unequal volumes):
        dC_i/dt = (C_in - C_i) / HRT_i
        where C_in = Co if i == 0 else C[i-1]

    Parameters:
        Co        : float       — inlet pulse concentration [mg/L]
        total_HRT : float       — total HRT across all tanks = Σ(V_i)/Q
        n         : int         — number of CSTRs in series
        volumes   : list        — individual tank volumes (unequal mode, requires Q)
        Q         : float       — flow rate (required for unequal volume mode)
        Cn        : float       — outlet concentration at time t
        t         : float or array — time(s) at which to evaluate C(t)

    Pass exactly ONE of (Co, total_HRT, Cn) as None to solve for it.
    n and t must always be provided.

    Returns:
        dict:
            "Co"          : float
            "total_HRT"   : float
            "HRT_per_tank": list of floats
            "n"           : int
            "volumes"     : list of floats
            "Q"           : float or None
            "t"           : array
            "C_t"         : array  — full concentration profile (plottable)
            "Cn"          : float  — concentration at final t
            "solved_for"  : str
            "solved_value": float
    """

    # --- Input validation ---
    if n is None:
        raise ValueError("n (number of reactors) must always be provided.")
    n = int(n)

    if t is None:
        raise ValueError("t (time) must always be provided.")
    t_array = np.atleast_1d(np.asarray(t, dtype=float))

    unequal_mode = volumes is not None

    if unequal_mode:
        if len(volumes) != n:
            raise ValueError(f"Length of volumes ({len(volumes)}) must equal n ({n}).")
        if Q is None:
            raise ValueError("Q (flow rate) is required when volumes are provided.")
        HRT_per_tank = [V / Q for V in volumes]
        _total_HRT   = sum(HRT_per_tank)
    else:
        if total_HRT is None:
            raise ValueError("total_HRT is required in equal-volume mode.")
        _total_HRT   = total_HRT
        HRT_per_tank = [_total_HRT / n] * n

    # -------------------------------------------------------------------------
    # ODE Solver (unequal volumes)
    # Each tank follows: dC_i/dt = (C_in - C_i) / HRT_i
    # Tanks are chained: first tank receives feed Co,
    # every other tank receives the outlet of the previous tank
    # -------------------------------------------------------------------------
    def run_ode_solver(Co_, t_eval):

        def tank_equations(time, C):
            dC = np.zeros(n)
            for i in range(n):
                C_in  = Co_   if i == 0 else C[i - 1]  # chain tanks together
                C_out = C[i]                             # current tank concentration
                dC[i] = (C_in - C_out) / HRT_per_tank[i]  # mass balance
            return dC

        C_initial = np.zeros(n)  # all tanks start empty

        solution = solve_ivp(
            fun    = tank_equations,
            t_span = (0, t_eval[-1]),
            y0     = C_initial,
            t_eval = t_eval,
            method = "RK45"
        )

        return solution.y[-1]  # outlet of last tank over time

    # -------------------------------------------------------------------------
    # Analytical solver (equal volumes)
    # C(t) = Co * (1/HRT_i) * (t/HRT_i)^(n-1) / (n-1)! * exp(-t/HRT_i)
    # -------------------------------------------------------------------------
    def run_analytical_solver(Co_, total_HRT_, t_eval):
        HRT_i = total_HRT_ / n
        return (
            Co_ * (1 / HRT_i)
            * (t_eval / HRT_i) ** (n - 1)
            / factorial(n - 1)
            * np.exp(-t_eval / HRT_i)
        )

    # --- Select mode and compute full C(t) profile ---
    def compute_C_profile(Co_, total_HRT_):
        if unequal_mode:
            return run_ode_solver(Co_, t_array)
        else:
            return run_analytical_solver(Co_, total_HRT_, t_array)

    # --- Identify unknowns ---
    unknowns = {
        "Co":        Co        is None,
        "total_HRT": total_HRT is None and not unequal_mode,
        "Cn":        Cn        is None,
    }
    num_unknowns = sum(unknowns.values())

    # If no unknown or only Cn is unknown, just return the full profile
    if num_unknowns == 0 or (unknowns["Cn"] and num_unknowns == 1):
        C_profile = compute_C_profile(Co, _total_HRT)
        return {
            "Co":           float(Co),
            "total_HRT":    float(_total_HRT),
            "HRT_per_tank": HRT_per_tank,
            "n":            n,
            "volumes":      volumes if volumes else [_total_HRT / n] * n,
            "Q":            Q,
            "t":            t_array,
            "C_t":          C_profile,
            "Cn":           float(C_profile[-1]),
            "solved_for":   "Cn",
            "solved_value": float(C_profile[-1]),
        }

    if num_unknowns != 1:
        raise ValueError(f"Exactly 1 of (Co, total_HRT, Cn) must be None (got {num_unknowns}).")

    # --- Solve for unknown using fsolve ---
    t_solve = float(t_array[0])

    def equation(x):
        _Co        = float(x[0]) if unknowns["Co"]        else Co
        _total_HRT = float(x[0]) if unknowns["total_HRT"] else _total_HRT if not unequal_mode else sum(HRT_per_tank)

        if unequal_mode:
            C_at_t = run_ode_solver(_Co, np.array([t_solve]))[0]
        else:
            C_at_t = run_analytical_solver(_Co, _total_HRT, np.array([t_solve]))[0]

        return C_at_t - Cn

    solution, info, ier, msg = fsolve(equation, [1.0], full_output=True)
    if ier != 1:
        raise RuntimeError(f"Solver failed to converge: {msg}")

    solved = float(solution[0])

    _Co        = Co        if not unknowns["Co"]        else solved
    _total_HRT = total_HRT if not unknowns["total_HRT"] else solved
    if unequal_mode:
        _total_HRT = sum(HRT_per_tank)

    C_profile = compute_C_profile(_Co, _total_HRT)

    return {
        "Co":           float(_Co),
        "total_HRT":    float(_total_HRT),
        "HRT_per_tank": HRT_per_tank,
        "n":            n,
        "volumes":      volumes if volumes else [_total_HRT / n] * n,
        "Q":            Q,
        "t":            t_array,
        "C_t":          C_profile,
        "Cn":           float(C_profile[-1]),
        "solved_for":   [k for k, v in unknowns.items() if v][0],
        "solved_value": float(solved),
    }


def cstr_series_sweep(sweep_param, sweep_values, **kwargs):
    """
    Sweeps one parameter and returns plottable arrays.

    Parameters:
        sweep_param  : str   — parameter to vary: 'n', 'total_HRT', 'Co'
        sweep_values : array — values to sweep over
        **kwargs     : all other cstr_series_calc parameters (fixed)

    Returns:
        dict:
            "sweep_values": 1D array
            "C_t"         : 2D array, shape (len(sweep_values), len(t))
            "t"           : 1D shared time axis
            "Co"          : 1D array
            "total_HRT"   : 1D array
            "n"           : 1D array
            "Cn"          : 1D array

    Example:
        t = np.linspace(0.01, 50, 500)
        results = cstr_series_sweep(
            sweep_param="n",
            sweep_values=[1, 2, 5, 10],
            Co=100, total_HRT=10, Cn=None, t=t
        )
        for i, n in enumerate([1, 2, 5, 10]):
            plt.plot(results["t"], results["C_t"][i], label=f"n={n}")
    """
    records = {k: [] for k in ["Co", "total_HRT", "n", "Cn", "C_t", "sweep_values"]}

    for val in sweep_values:
        params = dict(kwargs)
        params[sweep_param] = val
        result = cstr_series_calc(**params)

        records["sweep_values"].append(val)
        records["Co"].append(result["Co"])
        records["total_HRT"].append(result["total_HRT"])
        records["n"].append(result["n"])
        records["Cn"].append(result["Cn"])
        records["C_t"].append(result["C_t"])

    return {
        "sweep_values": np.array(records["sweep_values"]),
        "Co":           np.array(records["Co"]),
        "total_HRT":    np.array(records["total_HRT"]),
        "n":            np.array(records["n"]),
        "Cn":           np.array(records["Cn"]),
        "t":            kwargs["t"],                    # shared time axis
        "C_t":          np.array(records["C_t"]),       # shape: (n_sweep, n_t)
    }

In [ ]:
#new code

def f(x):
    return np.exp(-x**2)

#testing 


print("Hello World")

In [ ]:
print("hello world")

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# -----------------------------
# USER INPUTS
# -----------------------------
F = float(input("Enter flow rate F (L/s): "))
V = float(input("Enter reactor volume V (L): "))
Cin = float(input("Enter inlet concentration Cin (mol/L): "))
k = float(input("Enter reaction rate constant k (1/s): "))
n = float(input("Enter reaction order n: "))

# -----------------------------
# STEADY-STATE FUNCTION
# -----------------------------
def cstr_steady_state(C):
    return F * (Cin - C) - V * k * C**n

# -----------------------------
# SOLVE (try multiple guesses)
# -----------------------------
guesses = np.linspace(0, Cin, 5)
solutions = []

for guess in guesses:
    sol = fsolve(cstr_steady_state, guess)[0]
    if 0 <= sol <= Cin:  # physically meaningful
        solutions.append(sol)

# Remove duplicates
solutions = list(set([round(s, 5) for s in solutions]))

# -----------------------------
# OUTPUT RESULTS
# -----------------------------
print("\n--- Results ---")
for i, C in enumerate(solutions):
    X = (Cin - C) / Cin
    print(f"Solution {i+1}:")
    print(f"  Concentration: {C:.4f} mol/L")
    print(f"  Conversion: {X:.4f}")

# -----------------------------
# PLOT FUNCTION (visualize root)
# -----------------------------
C_vals = np.linspace(0, Cin, 100)
f_vals = cstr_steady_state(C_vals)

plt.figure()
plt.axhline(0)
plt.plot(C_vals, f_vals)
plt.xlabel("Concentration (C)")
plt.ylabel("Steady-State Function Value")
plt.title("CSTR Steady-State Equation")
plt.grid()
plt.show()

# -----------------------------
# PARAMETER SWEEP (vary k)
# -----------------------------
k_values = np.linspace(0.1, 5*k, 50)
C_results = []

for k_val in k_values:
    def func(C):
        return F * (Cin - C) - V * k_val * C**n
    
    C_sol = fsolve(func, Cin)[0]
    C_results.append(C_sol)

plt.figure()
plt.plot(k_values, C_results)
plt.xlabel("Reaction Rate Constant k")
plt.ylabel("Steady-State Concentration")
plt.title("Effect of k on Steady-State Concentration")
plt.grid()
plt.show()


from scipy.integrate import solve_ivp

# -----------------------------
# PFR MODEL
# -----------------------------
def pfr_model(V, C):
    return -k * C**n / F

# Solve from V = 0 → V = reactor volume
V_span = (0, V)
V_eval = np.linspace(0, V, 100)

solution = solve_ivp(pfr_model, V_span, [Cin], t_eval=V_eval)

V_vals = solution.t
C_vals = solution.y[0]

# -----------------------------
# OUTPUT
# -----------------------------
C_out = C_vals[-1]
X_pfr = (Cin - C_out) / Cin

print("\n--- PFR Results ---")
print(f"Outlet concentration: {C_out:.4f} mol/L")
print(f"Conversion: {X_pfr:.4f}")

ValueError: could not convert string to float: ''